# Notebook 3: Churn Risk Prediction 

In [ ]:
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
warnings.filterwarnings('ignore')


## 1. Load Data & Feature Engineering

In [ ]:
import sqlite3
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("Connecting to database...")
con = sqlite3.connect('data/olist.db')

query = '''
SELECT 
    o.order_id,
    c.customer_unique_id,
    r.review_score,
    r.review_comment_message,
    julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) as delivery_delay_days,
    julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) as actual_delivery_days,
    i.price,
    i.freight_value,
    p.product_photos_qty,
    p.product_description_lenght as description_length,
    c.customer_state,
    s.seller_state,
    p.product_category_name
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items i ON o.order_id = i.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN sellers s ON i.seller_id = s.seller_id
JOIN products p ON i.product_id = p.product_id
WHERE o.order_status = 'delivered'
'''
df_raw = pd.read_sql_query(query, con)
con.close()

# Feature Engineering
df = df_raw.copy()
df['freight_ratio'] = df['freight_value'] / df['price']
df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

# VADER Sentiment Analysis
print("Running VADER Sentiment Analysis on review comments...")
analyzer = SentimentIntensityAnalyzer()
# Update VADER lexicon with Portuguese churn keywords
pt_lexicon = {'atraso': -2.5, 'defeito': -3.0, 'cancelar': -2.5, 'péssimo': -3.5, 'ruim': -2.5, 'lixo': -3.0, 'falso': -3.0, 'quebrado': -2.5, 'demora': -2.0, 'nunca': -1.5, 'não': -1.0, 'ótimo': 2.5, 'bom': 2.0, 'excelente': 3.0, 'perfeito': 3.0}
analyzer.lexicon.update(pt_lexicon)
def get_vader_score(text):
    if pd.isna(text) or str(text).strip() == '':
        return 0.0
    return analyzer.polarity_scores(str(text))['compound']

df['review_vader_score'] = df['review_comment_message'].apply(get_vader_score)

# Label Encoding for categorical features
from sklearn.preprocessing import LabelEncoder
le_state = LabelEncoder()
df['customer_state_enc'] = le_state.fit_transform(df['customer_state'].astype(str))
le_category = LabelEncoder()
df['category_enc'] = le_category.fit_transform(df['product_category_name'].astype(str))

# Define Target: Churn Risk (Dissatisfied Experience = review score 1 or 2)
df['churn_risk'] = (df['review_score'] <= 2).astype(int)

# Clean up NaNs
FEATURES = ['delivery_delay_days', 'actual_delivery_days', 'price', 'freight_ratio', 
            'product_photos_qty', 'description_length', 'same_state', 
            'customer_state_enc', 'category_enc', 'review_vader_score']
df = df.dropna(subset=FEATURES)

print(f"Dataset built: {len(df):,} orders | At-risk rate: {df['churn_risk'].mean()*100:.1f}%")




## 2. Train XGBoost Model & Cross-Validation

In [ ]:
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import numpy as np

X_raw = df[FEATURES].astype(float)
y = df['churn_risk'].values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

n0, n1 = (y == 0).sum(), (y == 1).sum()
clf_params = {
    'n_estimators': 400,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 10,
    'scale_pos_weight': round(n0 / n1),
    'eval_metric': 'auc',
    'objective': 'binary:logistic',
    'random_state': 42,
    'n_jobs': -1
}

print("Running 5-fold cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for train_idx, val_idx in cv.split(X, y):
    model = XGBClassifier(**clf_params)
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict_proba(X[val_idx])[:, 1]
    cv_scores.append(roc_auc_score(y[val_idx], preds))

cvs = np.array(cv_scores)
print(f"CV AUC:  {cvs.mean():.4f} +/- {cvs.std():.4f}")

model = XGBClassifier(**clf_params)
model.fit(X, y)
probs = model.predict_proba(X)[:, 1]
print(f"Train AUC (full): {roc_auc_score(y, probs):.4f}")

# Save the model and artifacts
print("Saving model artifacts...")
import pickle
with open('models/xgb_churn_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('models/feature_names.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

# Also save predictions to outputs/ so the Streamlit dashboard works
print("Saving predictions to outputs/churn_predictions.csv...")
out_df = df.copy()
out_df['churn_prob'] = probs
out_df.to_csv('outputs/churn_predictions.csv', index=False)
print("Done!")



## 3. SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X_raw, plot_type="bar", show=False)
plt.title('SHAP Feature Importance (Churn Risk)')
plt.tight_layout()
